# Đồ Án Trí Tuệ Nhân Tạo: 8-Puzzle


**Source code và Tài liệu đầy đủ:** [GitHub - 8_puzzle_project](https://github.com/24110207-Hai-Nguyen/8_puzzle_project)


In [ ]:
# BẢN GỘP TOÀN BỘ CODE 8-PUZZLE VÀO 1 FILE DUY NHẤT
import tkinter as tk
from tkinter import ttk
import threading
import time
import random
import heapq
from collections import deque
import math
import sys


# ========================================
# FILE: map_generator.py
# ========================================

from itertools import permutations

GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)

def count_inversions(state):
    inv = 0
    for i in range(9):
        for j in range(i+1, 9):
            if state[i] != 0 and state[j] != 0 and state[i] > state[j]:
                inv += 1
    return inv

def is_solvable(state):
    return count_inversions(state) % 2 == 0

def get_neighbors(state):
    idx = state.index(0)
    row, col = divmod(idx, 3)
    moves = [(-1, 0, 'Up'), (1, 0, 'Down'), (0, -1, 'Left'), (0, 1, 'Right')]
    neighbors = []
    for dr, dc, move in moves:
        r, c = row + dr, col + dc
        if 0 <= r < 3 and 0 <= c < 3:
            new_idx = r * 3 + c
            new_state = list(state)
            new_state[idx], new_state[new_idx] = new_state[new_idx], new_state[idx]
            neighbors.append((tuple(new_state), move))
    return neighbors

def generate_random_solvable_states(num):
    states = []
    while len(states) < num:
        state = list(GOAL)
        for _ in range(30):
            nbs = get_neighbors(tuple(state))
            state = list(random.choice(nbs)[0])
        if is_solvable(state) and tuple(state) not in states:
            states.append(tuple(state))
    return states

def format_state_matrix(state):
    rows = []
    for i in range(3):
        row = [str(x) if x != 0 else "_" for x in state[i*3:(i+1)*3]]
        rows.append(" " + " ".join(row))
    return "\n".join(rows)

def heuristic(state):
    return sum(1 for i in range(9) if state[i] != 0 and state[i] != GOAL[i])

def value(state):
    return 9 - heuristic(state)

# ========================================
# FILE: algorithms/base.py
# ========================================


class Node:
    _id_counter = 65
    def __init__(self, state, parent=None, action=None, cost=0, depth=0, h_cost=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.cost = cost
        self.depth = depth
        self.h_cost = h_cost
        self.f_cost = cost + h_cost
        if parent is None:
            self.name = "A"
        else:
            self.name = chr(Node._id_counter)
            Node._id_counter += 1
            if Node._id_counter > 90:
                Node._id_counter = 65

    def __lt__(self, other):
        return self.f_cost < other.f_cost

class PuzzleSearchBase:
    """Base class for puzzle algorithms."""
    def __init__(self, start_state, param_val=35):
        self.start_state = start_state
        self.param_val = param_val
        self.explored = set()
        self.frontier = []
        self.result_node = None
        self.nodes_log = []
        self.step_log = []

        Node._id_counter = 65

    def solve(self, log_callback=None, cancel_event=None):
        raise NotImplementedError

class ComplexProblem:
    def __init__(self, initial_belief, goal_test, actions_func, results_func):
        self.initial_belief = initial_belief
        self.goal_test = goal_test
        self.actions = actions_func
        self.results = results_func

    def is_goal(self, state):
        return self.goal_test(state)

# ========================================
# FILE: ui/map_canvas.py
# ========================================


class PuzzleCanvas(tk.Frame):
    def __init__(self, master, **kwargs):
        super().__init__(master, bg="#FFFFFF", **kwargs)
        self.grid_frames = []
        self.cells = [] # list of lists of labels

        self.turn_label = tk.Label(self, text="", font=("Segoe UI", 12, "bold"), bg="#FFFFFF", fg="#1E293B")
        self.turn_label.pack(pady=(0, 5))

        self.container = tk.Frame(self, bg="#FFFFFF")
        self.container.pack(expand=True)

    def set_turn(self, action_str):
        if not action_str:
            self.turn_label.config(text="")
            return

        if "[MAX]" in action_str:
            self.turn_label.config(text="MAX'S TURN", fg="#2563EB")
        elif "[MIN]" in action_str:
            self.turn_label.config(text="MIN'S TURN", fg="#EF4444")
        elif "[CHANCE]" in action_str:
            self.turn_label.config(text="CHANCE'S TURN", fg="#F59E0B")
        else:
            self.turn_label.config(text="")

    def _build_grids(self, num_grids):
        for f in self.grid_frames:
            f.destroy()
        self.grid_frames = []
        self.cells = []

        # If there are many grids, scale down the font so they fit
        font_size = 24 if num_grids == 1 else 16
        width = 4 if num_grids == 1 else 3
        height = 2 if num_grids == 1 else 1

        for g in range(num_grids):
            frame = tk.Frame(self.container, bg="#FFFFFF")
            frame.pack(side="left", padx=10)

            if num_grids > 1:
                tk.Label(frame, text=f"State {g+1}", bg="#FFFFFF", fg="#475569", font=("Segoe UI", 10, "bold")).pack(pady=(0, 5))

            grid_f = tk.Frame(frame, bg="#CBD5E1", padx=2, pady=2)
            grid_f.pack()

            grid_cells = []
            for i in range(9):
                lbl = tk.Label(
                    grid_f,
                    text="",
                    font=("Segoe UI", font_size, "bold"),
                    width=width, height=height,
                    bg="#2563EB", fg="#FFFFFF",
                    bd=0, highlightthickness=0
                )
                lbl.grid(row=i//3, column=i%3, padx=2, pady=2)
                grid_cells.append(lbl)

            self.grid_frames.append(frame)
            self.cells.append(grid_cells)

    def update_grid(self, state_or_belief):
        if not state_or_belief: return

        # Check if single state or belief state
        is_belief = isinstance(state_or_belief[0], (list, tuple))

        states = state_or_belief if is_belief else [state_or_belief]

        if len(states) != len(self.grid_frames):
            self._build_grids(len(states))

        colors = ["#2563EB", "#7C3AED", "#0891B2"] # Blue, Purple, Cyan for different states

        for g, state in enumerate(states):
            primary_color = colors[g % len(colors)] if is_belief else "#2563EB"
            for i, val in enumerate(state):
                if val == 0 or val == -1:
                    text = "?" if val == -1 else ""
                    self.cells[g][i].config(text=text, bg="#E5E7EB", fg="#1E293B" if val == -1 else "#FFFFFF")
                else:
                    self.cells[g][i].config(text=str(val), bg=primary_color, fg="#FFFFFF")

# ========================================
# FILE: ui/stats_panel.py
# ========================================


class StatsPanel(tk.Frame):
    def __init__(self, master, **kwargs):
        super().__init__(master, bg="#FFFFFF", **kwargs)
        self._build()

    def _build(self):
        self.labels = {}
        fields = [
            ("step",      "Step:"),
            ("algo",      "Algorithm:"),
            ("expanded",  "Expanded:"),
            ("action",    "Action:"),
            ("path_cost", "Path Cost:"),
            ("depth",     "Depth:"),
            ("ids_limit", "IDS Limit:"),
            ("frontier",  "Frontier:"),
            ("explored",  "Explored:"),
            ("status",    "Status:"),
        ]

        for key, label in fields:
            row = tk.Frame(self, bg="#FFFFFF")
            row.pack(fill="x", padx=10, pady=4)
            tk.Label(row, text=label, bg="#FFFFFF", fg="#475569",
                     font=("Segoe UI", 10), width=12, anchor="w").pack(side="left")
            default_val = "Ready." if key == "status" else "-"
            lbl = tk.Label(row, text=default_val, bg="#FFFFFF", fg="#1E293B",
                           font=("Segoe UI", 10, "italic" if key=="status" else "normal"), anchor="w")
            lbl.pack(side="left")
            self.labels[key] = lbl

    def update_stat(self, key, value):
        if key in self.labels:
            self.labels[key].config(text=str(value))

    def update_status(self, text, color="#2563EB"):
        self.labels["status"].config(text=text, fg=color)

    def reset_current(self):
        for key, lbl in self.labels.items():
            if key == "status":
                lbl.config(text="Ready.", fg="#1E293B")
            else:
                lbl.config(text="-", fg="#1E293B")

# ========================================
# FILE: ui/details_notebook.py
# ========================================


class MiniPuzzle(tk.Frame):
    def __init__(self, master, state, step_idx, action, click_cb, **kwargs):
        super().__init__(master, bg="#FFFFFF", **kwargs)
        self.step_idx = step_idx
        self.click_cb = click_cb

        def on_click(e):
            if self.click_cb:
                self.click_cb(self.step_idx)

        self.bind("<Button-1>", on_click)

        lbl_title = tk.Label(self, text=f"Step {step_idx}\n{action}", bg="#FFFFFF", fg="#0891B2", font=("Segoe UI", 9, "bold"), cursor="hand2")
        lbl_title.pack(pady=(2, 2))
        lbl_title.bind("<Button-1>", on_click)

        is_belief = isinstance(state[0], (list, tuple))
        states = state if is_belief else [state]

        container = tk.Frame(self, bg="#FFFFFF")
        container.pack(pady=(0, 4), padx=2)

        for st in states:
            grid_f = tk.Frame(container, bg="#CBD5E1", padx=2, pady=2, cursor="hand2")
            grid_f.pack(side="left", padx=2)
            grid_f.bind("<Button-1>", on_click)

            for i, val in enumerate(st):
                text = str(val) if val not in (0, -1) else ("?" if val == -1 else "")
                bg_color = "#1E3A8A" if val != 0 else "#F1F5F9"
                lbl = tk.Label(grid_f, text=text, width=2, height=1, font=("Segoe UI", 8, "bold"), bg=bg_color, fg="#FFFFFF", cursor="hand2")
                lbl.grid(row=i//3, column=i%3, padx=1, pady=1)
                lbl.bind("<Button-1>", on_click)


class DetailsNotebook(tk.Frame):
    def __init__(self, master, on_solution_click_cb, **kwargs):
        super().__init__(master, bg="#F3F4F6", **kwargs)
        self.on_solution_click_cb = on_solution_click_cb
        self._build()

    def _build(self):
        style = ttk.Style()
        style.theme_use("clam")
        style.configure("TNotebook", background="#F3F4F6", borderwidth=0)
        style.configure("TNotebook.Tab", background="#E5E7EB", foreground="#475569", padding=[10, 2])
        style.map("TNotebook.Tab", background=[("selected", "#FFFFFF")], foreground=[("selected", "#0891B2")])

        self.notebook = ttk.Notebook(self)
        self.notebook.pack(fill=tk.BOTH, expand=True)

        self.tabs = {}
        tab_names = ["Nodes", "Frontier", "Explored", "Step Log", "Conditional Plan", "Solution Path"]

        for name in tab_names:
            txt = tk.Text(self.notebook, font=("Consolas", 9), bg="#FFFFFF", fg="#1E293B", bd=0, insertbackground="#1E293B")
            self.notebook.add(txt, text=name)
            self.tabs[name] = txt

        self.tabs["Solution Path"].config(wrap="word", cursor="arrow")

    def clear_all(self):
        for txt in self.tabs.values():
            txt.config(state="normal")
            txt.delete('1.0', tk.END)

    def rename_tab_text(self, internal_key, new_text):
        if internal_key in self.tabs:
            idx = list(self.tabs.keys()).index(internal_key)
            self.notebook.tab(idx, text=new_text)

    def write_tab(self, tab_name, text):
        if tab_name in self.tabs:
            txt = self.tabs[tab_name]
            txt.config(state="normal")
            txt.insert(tk.END, text + "\n")
            txt.see(tk.END)

    def render_solution_path(self, path):
        txt = self.tabs["Solution Path"]
        txt.config(state="normal")
        txt.delete("1.0", tk.END)

        display_path = path
        skipped = False
        if len(path) > 200:
            display_path = path[:100] + [None] + path[-100:]
            skipped = True

        for display_idx, node in enumerate(display_path):
            if node is None:
                lbl = tk.Label(txt, text=f" ... \n(Skipped {len(path) - 200} steps)\n ... ", bg="#FFFFFF", fg="#475569", font=("Segoe UI", 10, "italic"))
                txt.window_create(tk.END, window=lbl)
                lbl = tk.Label(txt, text=" ➔ ", bg="#FFFFFF", fg="#475569", font=("Segoe UI", 12))
                txt.window_create(tk.END, window=lbl)
                continue

            # Find the actual index in the original path (safe because node instances are unique in the path)
            actual_idx = path.index(node) if node in path else display_idx
            act = getattr(node, 'action', "Start") if getattr(node, 'action', None) else "Start"
            mini = MiniPuzzle(txt, node.state, actual_idx, act, self.on_solution_click_cb)
            txt.window_create(tk.END, window=mini)

            if display_idx < len(display_path) - 1:
                lbl = tk.Label(txt, text=" ➔ ", bg="#FFFFFF", fg="#475569", font=("Segoe UI", 12))
                txt.window_create(tk.END, window=lbl)

        txt.config(state="disabled")

    def highlight_step(self, step_idx):
        # With graphical tiles, highlighting would mean iterating through embedded widgets
        # For simplicity, we just pass since the canvas updates visually.
        pass

# ========================================
# FILE: algorithms/uninformed/bfs.py
# ========================================

import collections

class BFS(PuzzleSearchBase):
    def solve(self, log_callback=None, cancel_event=None):
        root_node = Node(self.start_state, cost=0, depth=0)
        queue = collections.deque([root_node])
        self.explored.add(self.start_state)

        while queue:
            if cancel_event and cancel_event.is_set():
                break

            node = queue.popleft()
            if log_callback: log_callback(f"Expanded: {node.name}")

            if node.state == GOAL:
                self.result_node = node
                break

            for state, action in get_neighbors(node.state):
                if state not in self.explored and node.depth < self.param_val:
                    self.explored.add(state)
                    child = Node(state, node, action, node.cost + 1, node.depth + 1)
                    queue.append(child)
                    if log_callback:
                        log_callback(f"Node: {child.name} | parent={node.name} | action={action} | depth={child.depth}\n{format_state_matrix(state)}")

        self.frontier = list(queue)
        return self.result_node, self.frontier, self.explored

# ========================================
# FILE: algorithms/uninformed/dfs.py
# ========================================


class DFS(PuzzleSearchBase):
    def solve(self, log_callback=None, cancel_event=None):
        root_node = Node(self.start_state, cost=0, depth=0)
        stack = [root_node]
        self.explored = set()
        visited_depth = {self.start_state: 0}

        while stack:
            if cancel_event and cancel_event.is_set():
                break

            node = stack.pop()
            self.explored.add(node.state)
            if log_callback: log_callback(f"Expanded: {node.name}")

            if node.state == GOAL:
                self.result_node = node
                break

            if node.depth < self.param_val:
                for state, action in get_neighbors(node.state):
                    if state not in visited_depth or visited_depth[state] > node.depth + 1:
                        visited_depth[state] = node.depth + 1
                        child = Node(state, node, action, node.cost + 1, node.depth + 1)
                        stack.append(child)
                        if log_callback:
                            log_callback(f"Node: {child.name} | parent={node.name} | action={action} | depth={child.depth}\n{format_state_matrix(state)}")

        self.frontier = stack
        return self.result_node, self.frontier, self.explored

# ========================================
# FILE: algorithms/informed/ucs.py
# ========================================


class UCS(PuzzleSearchBase):
    def solve(self, log_callback=None, cancel_event=None):
        root_node = Node(self.start_state, cost=0, depth=0)
        pq = [(root_node.cost, id(root_node), root_node)]

        while pq:
            if cancel_event and cancel_event.is_set():
                break

            cost, _, node = heapq.heappop(pq)
            if log_callback: log_callback(f"Expanded: {node.name}")

            if node.state == GOAL:
                self.result_node = node
                break

            if node.state not in self.explored:
                self.explored.add(node.state)
                if node.depth < self.param_val:
                    for state, action in get_neighbors(node.state):
                        if state not in self.explored:
                            child = Node(state, node, action, node.cost + 1, node.depth + 1)
                            heapq.heappush(pq, (child.cost, id(child), child))
                            if log_callback:
                                log_callback(f"Node: {child.name} | parent={node.name} | action={action} | cost={child.cost}\n{format_state_matrix(state)}")

        self.frontier = [n for _, _, n in pq]
        return self.result_node, self.frontier, self.explored

# ========================================
# FILE: algorithms/uninformed/ids.py
# ========================================


class IDS(PuzzleSearchBase):
    def solve(self, log_callback=None, cancel_event=None):
        max_depth_limit = self.param_val
        final_explored = set()
        root_node = Node(self.start_state, cost=0, depth=0)

        for depth_limit in range(1, max_depth_limit + 1):
            if cancel_event and cancel_event.is_set():
                break

            if log_callback: log_callback(f"StepLog: Trying depth limit = {depth_limit}")
            visited_depth = {self.start_state: 0}
            stack = [(root_node, 0)]
            found = False

            while stack:
                if cancel_event and cancel_event.is_set():
                    break

                node, d = stack.pop()
                final_explored.add(node.state)

                if node.state == GOAL:
                    self.result_node = node
                    found = True
                    break

                if d < depth_limit:
                    for state, action in get_neighbors(node.state):
                        if state not in visited_depth or visited_depth[state] > d + 1:
                            visited_depth[state] = d + 1
                            child = Node(state, node, action, node.cost + 1, d + 1)
                            stack.append((child, d + 1))
                            if log_callback:
                                log_callback(f"Node: {child.name} | parent={node.name} | action={action} | d={d+1}\n{format_state_matrix(state)}")

            if found:
                break

        self.explored = final_explored
        self.frontier = [n for n, _ in stack] if not self.result_node else []
        return self.result_node, self.frontier, self.explored

# ========================================
# FILE: algorithms/informed/gbfs.py
# ========================================


class GreedyBestFirstSearch(PuzzleSearchBase):
    def solve(self, log_callback=None, cancel_event=None):
        h = heuristic(self.start_state)
        root_node = Node(self.start_state, cost=0, depth=0, h_cost=h)
        frontier = [(root_node.h_cost, id(root_node), root_node)]
        reached = {self.start_state: root_node}

        while frontier:
            if cancel_event and cancel_event.is_set():
                break

            _, _, node = heapq.heappop(frontier)
            if log_callback: log_callback(f"Expanded: {node.name}")

            if node.state == GOAL:
                self.result_node = node
                break

            for state, action in get_neighbors(node.state):
                h_val = heuristic(state)
                child = Node(state, node, action, node.cost + 1, node.depth + 1, h_cost=h_val)
                if state not in reached:
                    reached[state] = child
                    heapq.heappush(frontier, (child.h_cost, id(child), child))
                    if log_callback:
                        log_callback(f"Node: {child.name} | parent={node.name} | h={child.h_cost}\n{format_state_matrix(state)}")

        self.frontier = [n for _, _, n in frontier]
        self.explored = set(reached.keys())
        return self.result_node, self.frontier, self.explored

# ========================================
# FILE: algorithms/informed/astar.py
# ========================================


class AStar(PuzzleSearchBase):
    def solve(self, log_callback=None, cancel_event=None):
        h = heuristic(self.start_state)
        root_node = Node(self.start_state, cost=0, depth=0, h_cost=h)
        frontier = [(root_node.f_cost, id(root_node), root_node)]
        reached = {self.start_state: root_node}

        while frontier:
            if cancel_event and cancel_event.is_set():
                break

            _, _, node = heapq.heappop(frontier)
            if log_callback: log_callback(f"Expanded: {node.name}")

            if node.state == GOAL:
                self.result_node = node
                break

            for state, action in get_neighbors(node.state):
                g_new = node.cost + 1
                h_val = heuristic(state)

                if state in reached:
                    old_node = reached[state]
                    if g_new < old_node.cost:
                        old_node.cost = g_new
                        old_node.h_cost = h_val
                        old_node.f_cost = g_new + h_val
                        old_node.parent = node
                        old_node.action = action
                        old_node.depth = node.depth + 1
                        heapq.heappush(frontier, (old_node.f_cost, id(old_node), old_node))
                else:
                    child = Node(state, node, action, g_new, node.depth + 1, h_cost=h_val)
                    reached[state] = child
                    heapq.heappush(frontier, (child.f_cost, id(child), child))
                    if log_callback:
                        log_callback(f"Node: {child.name} | g={child.cost} h={child.h_cost} f={child.f_cost}\n{format_state_matrix(state)}")

        self.frontier = [n for _, _, n in frontier]
        self.explored = set(reached.keys())
        return self.result_node, self.frontier, self.explored

# ========================================
# FILE: algorithms/local_search/simple_hill_climbing.py
# ========================================


class SimpleHillClimbing(PuzzleSearchBase):
    def solve(self, log_callback=None, cancel_event=None):
        current_node = Node(self.start_state, cost=0, depth=0)
        if log_callback:
            log_callback(f"StepLog: Start: {format_state_matrix(current_node.state)} (Value={value(current_node.state)})")

        while True:
            if cancel_event and cancel_event.is_set():
                break

            if current_node.state == GOAL:
                self.result_node = current_node
                break

            found_better = False
            for state, action in get_neighbors(current_node.state):
                if log_callback:
                    log_callback(f"Node: Neighbor: action={action}, value={value(state)}\n{format_state_matrix(state)}")
                if value(state) > value(current_node.state):
                    current_node = Node(state, current_node, action, current_node.cost+1, current_node.depth+1)
                    if log_callback:
                        log_callback(f"StepLog:   -> Move to (Value={value(state)})")
                    found_better = True
                    break

            if not found_better:
                if log_callback: log_callback(f"StepLog: Local maximum at {current_node.name}")
                self.result_node = current_node
                break

        return self.result_node, [], set()

# ========================================
# FILE: algorithms/local_search/steepest_ascent_hill_climbing.py
# ========================================


class SteepestAscentHillClimbing(PuzzleSearchBase):
    def solve(self, log_callback=None, cancel_event=None):
        current_node = Node(self.start_state, cost=0, depth=0)
        if log_callback:
            log_callback(f"StepLog: Start: {format_state_matrix(current_node.state)} (Value={value(current_node.state)})")

        while True:
            if cancel_event and cancel_event.is_set():
                break

            if current_node.state == GOAL:
                self.result_node = current_node
                break

            best_neighbor = None
            best_value = value(current_node.state)

            for state, action in get_neighbors(current_node.state):
                v = value(state)
                if log_callback:
                    log_callback(f"Node: Neighbor: action={action}, value={v}\n{format_state_matrix(state)}")
                if v > best_value:
                    best_value = v
                    best_neighbor = (state, action)

            if best_neighbor:
                state, action = best_neighbor
                current_node = Node(state, current_node, action, current_node.cost+1, current_node.depth+1)
                if log_callback:
                    log_callback(f"StepLog:   -> Move to best (Value={best_value})")
            else:
                if log_callback: log_callback(f"StepLog: Local maximum at {current_node.name}")
                self.result_node = current_node
                break

        return self.result_node, [], set()

# ========================================
# FILE: algorithms/local_search/stochastic_hill_climbing.py
# ========================================


class StochasticHillClimbing(PuzzleSearchBase):
    def solve(self, log_callback=None, cancel_event=None):
        current_node = Node(self.start_state, cost=0, depth=0)
        if log_callback:
            log_callback(f"StepLog: Start: {format_state_matrix(current_node.state)} (Value={value(current_node.state)})")

        while True:
            if cancel_event and cancel_event.is_set():
                break

            if current_node.state == GOAL:
                self.result_node = current_node
                break

            better_neighbors = []
            current_value = value(current_node.state)

            for state, action in get_neighbors(current_node.state):
                v = value(state)
                if log_callback:
                    log_callback(f"Node: Neighbor: action={action}, value={v}\n{format_state_matrix(state)}")
                if v > current_value:
                    better_neighbors.append((state, action, v))

            if better_neighbors:
                # Stochastic: Pick randomly among better neighbors
                state, action, v = random.choice(better_neighbors)
                current_node = Node(state, current_node, action, current_node.cost+1, current_node.depth+1)
                if log_callback:
                    log_callback(f"StepLog:   -> Move randomly to (Value={v})")
            else:
                if log_callback: log_callback(f"StepLog: Local maximum at {current_node.name}")
                self.result_node = current_node
                break

        return self.result_node, [], set()

# ========================================
# FILE: algorithms/local_search/simulated_annealing.py
# ========================================


class SimulatedAnnealing(PuzzleSearchBase):
    def solve(self, log_callback=None, cancel_event=None):
        max_iters = int(getattr(self, 'param_val', 100)) * 100
        if max_iters < 100: max_iters = 10000

        max_restarts = 15

        for attempt in range(max_restarts):
            current_node = Node(self.start_state, cost=0, depth=0)

            for t in range(1, max_iters + 1):
                if cancel_event and cancel_event.is_set():
                    self.result_node = current_node
                    return self.result_node, [], set()

                T = float(max_iters) / t
                if T == 0:
                    break

                if current_node.state == GOAL:
                    self.result_node = current_node
                    if log_callback and attempt > 0:
                        log_callback(f"StepLog: Goal found on restart attempt {attempt+1}!")
                    return self.result_node, [], set()

                neighbors = get_neighbors(current_node.state)
                next_state, action = random.choice(neighbors)
                if log_callback and attempt == 0 and t < 10:
                    log_callback(f"Node: Neighbor: action={action}, value={value(next_state)}\n{format_state_matrix(next_state)}")

                delta_e = value(next_state) - value(current_node.state)
                if delta_e > 0 or random.random() < math.exp(delta_e / T):
                    current_node = Node(next_state, current_node, action, current_node.cost+1, current_node.depth+1)
                    if log_callback and attempt == 0 and t < 10:
                        log_callback(f"StepLog:   -> Move to {action} (Value={value(next_state)})")

            if log_callback:
                log_callback(f"StepLog: Local max reached, restarting (attempt {attempt+1}/{max_restarts})...")

        self.result_node = current_node
        return self.result_node, [], set()

# ========================================
# FILE: algorithms/local_search/local_beam.py
# ========================================


class LocalBeamSearch(PuzzleSearchBase):
    def solve(self, log_callback=None, cancel_event=None):
        k = self.param_val
        if log_callback: log_callback(f"StepLog: Beam width k = {k}")

        current_states = [Node(self.start_state, cost=0, depth=0)]

        if log_callback:
            log_callback(f"StepLog: Initial beam:")
            for ns in current_states:
                log_callback(f"StepLog:   {ns.name} (Value={value(ns.state)})")

        visited = {self.start_state}
        max_iters = 100
        iters = 0

        while iters < max_iters:
            iters += 1
            if cancel_event and cancel_event.is_set():
                break

            all_neighbors = []
            found_goal = None

            for node in current_states:
                for state, action in get_neighbors(node.state):
                    child = Node(state, node, action, node.cost+1, node.depth+1)
                    all_neighbors.append(child)
                    if state == GOAL:
                        found_goal = child
                        break
                if found_goal: break

            if found_goal:
                self.result_node = found_goal
                break

            all_neighbors.sort(key=lambda n: value(n.state), reverse=True)

            # Filter out visited states to prevent trivial loops
            unvisited_neighbors = [n for n in all_neighbors if n.state not in visited]

            if not unvisited_neighbors:
                if log_callback: log_callback("StepLog: Dead end reached (all neighbors visited).")
                self.result_node = current_states[0]
                break

            next_states = unvisited_neighbors[:k]
            for ns in next_states:
                visited.add(ns.state)

            if log_callback:
                log_callback(f"StepLog: Next beam (Iter {iters}):")
                for ns in next_states:
                    log_callback(f"StepLog:   {ns.name} (Value={value(ns.state)})")
                    log_callback(f"Node: Beam item: {ns.name}\n{format_state_matrix(ns.state)}")

            current_states = next_states
            self.result_node = current_states[0] # Keep track of the best node so far

        return self.result_node, [], set()

# ========================================
# FILE: algorithms/complex/belief_state_bfs.py
# ========================================


class BeliefNode:
    def __init__(self, state, parent=None, action=None, cost=0):
        self.state = state  # state is a tuple of 8-puzzle states (belief state)
        self.parent = parent
        self.action = action
        self.cost = cost
        self.depth = 0 if parent is None else parent.depth + 1

def apply_action(single_state, action):
    # Standard actions: Up, Down, Left, Right
    for n_state, n_action in get_neighbors(single_state):
        if n_action == action:
            return n_state

    # If action is illegal, return the same state (wall-bumping semantics)
    return single_state

class BeliefStateBFS:
    def __init__(self, start_belief_state, param_val=None):
        # Keep duplicates so the UI shows a constant number of boards
        self.start_belief_state = tuple(sorted(start_belief_state))

    def solve(self, log_callback=None, cancel_event=None):
        start_node = Node(self.start_belief_state)

        # Check if goal is achieved for ALL states in the initial belief state
        if all(s == GOAL for s in self.start_belief_state):
            return start_node, [], set()

        frontier = deque([start_node])
        explored = set()
        explored.add(self.start_belief_state)

        actions = ['Up', 'Down', 'Left', 'Right']

        nodes_expanded = 0

        while frontier:
            if cancel_event and cancel_event.is_set():
                return None, list(frontier), explored

            node = frontier.popleft()
            nodes_expanded += 1

            if log_callback and nodes_expanded % 100 == 0:
                log_callback(f"StepLog: Expanded {nodes_expanded} belief states...")

            for action in actions:
                # Apply action without removing duplicates to maintain board count in UI
                next_belief_list = [apply_action(s, action) for s in node.state]
                next_belief = tuple(sorted(next_belief_list))

                if next_belief not in explored:
                    # Collapse to 1 state only when reaching the goal
                    if all(s == GOAL for s in next_belief):
                        child = BeliefNode((GOAL,), node, action, node.cost + 1)
                        if log_callback:
                            log_callback(f"StepLog: Goal found after {nodes_expanded} expansions!")
                        return child, list(frontier), explored

                    child = BeliefNode(next_belief, node, action, node.cost + 1)
                    explored.add(next_belief)
                    frontier.append(child)

        return None, list(frontier), explored

# ========================================
# FILE: algorithms/complex/partial_observation.py
# ========================================

import itertools

def generate_belief_state(partial_state):
    # partial_state is a tuple where unknown tiles are -1
    # Example: (1, 2, 3, 4, 5, 6, -1, -1, 0)

    missing_tiles = set(range(9)) - set(partial_state)
    missing_tiles.discard(-1)

    missing_tiles = list(missing_tiles)
    unknown_indices = [i for i, val in enumerate(partial_state) if val == -1]

    belief_state = []

    for perm in itertools.permutations(missing_tiles):
        state = list(partial_state)
        for idx, val in zip(unknown_indices, perm):
            state[idx] = val
        if is_solvable(state):
            belief_state.append(tuple(state))

    return tuple(belief_state)

class PartialObservationSearch:
    def __init__(self, partial_state, param_val=None):
        self.start_belief_state = generate_belief_state(partial_state)
        self.bfs_solver = BeliefStateBFS(self.start_belief_state, param_val)

    def solve(self, log_callback=None, cancel_event=None):
        if log_callback:
            log_callback(f"StepLog: Initial partial state: {self.start_belief_state}")
        return self.bfs_solver.solve(log_callback, cancel_event)

# ========================================
# FILE: algorithms/complex/and_or_search.py
# ========================================


class DummyNode:
    def __init__(self, state):
        self.state = state
        self.parent = None
        self.action = "Conditional Plan (See Tab)"
        self.cost = 0
        self.depth = 0

def results(state, action):
    # Non-deterministic action:
    # 1. Action succeeds
    # 2. Action fails (slippery, state remains the same)
    for n_state, n_action in get_neighbors(state):
        if n_action == action:
            return {"success": n_state, "fail": state}
    return {"fail": state}

class AndOrGraphSearch:
    def __init__(self, start_state, param_val=None):
        self.start_state = start_state
        if isinstance(start_state[0], (list, tuple)):
            self.start_state = start_state[0] # Take first if it's a belief state
        self.max_depth = param_val if param_val else 5
        self.log = None

    def solve(self, log_callback=None, cancel_event=None):
        self.log = log_callback

        plan = self.or_search(self.start_state, [], 0, cancel_event)

        if plan == "failure":
            if log_callback: log_callback("Conditional Plan: FAILURE (Depth limit reached or no path)")
            return None, [], []

        # Format the plan
        plan_str = self.format_plan(plan)
        if log_callback:
            log_callback(f"Conditional Plan:\n{plan_str}")

        dummy = DummyNode(GOAL)
        dummy.action = "Plan Generated (See Conditional Plan tab)"
        return dummy, [], []

    def or_search(self, state, path, depth, cancel_event):
        if cancel_event and cancel_event.is_set():
            return "failure"
        if state == GOAL:
            return []
        if state in path:
            return f"LOOP (Retry action)"
        if depth >= self.max_depth:
            return "failure"

        for action in ['Up', 'Down', 'Left', 'Right']:
            next_states_dict = results(state, action)

            # If the action is illegal, it only returns fail, which is a guaranteed infinite loop.
            if len(next_states_dict) == 1 and "fail" in next_states_dict:
                continue

            plan = self.and_search(next_states_dict, path + [state], depth + 1, cancel_event)
            if plan != "failure":
                return [action, plan]
        return "failure"

    def and_search(self, states_dict, path, depth, cancel_event):
        if cancel_event and cancel_event.is_set():
            return "failure"
        plan_dict = {}
        for outcome, s in states_dict.items():
            plan = self.or_search(s, path, depth, cancel_event)
            if plan == "failure":
                return "failure"
            plan_dict[outcome] = plan
        return plan_dict

    def format_plan(self, plan, indent=""):
        if not plan:
            return indent + "Goal Reached!\n"
        if isinstance(plan, str):
            # This handles "LOOP (Retry action)"
            return indent + plan + "\n"
        if isinstance(plan, list):
            action, and_plan = plan
            res = indent + f"Action: {action}\n"
            res += self.format_plan(and_plan, indent + "  ")
            return res
        if isinstance(plan, dict):
            res = ""
            for outcome, subplan in plan.items():
                if outcome == "success":
                    res += indent + f"If tile moves successfully:\n"
                else:
                    res += indent + f"If state remains unchanged:\n"
                res += self.format_plan(subplan, indent + "  ")
            return res
        return ""

# ========================================
# FILE: algorithms/csp/csp_solver.py
# ========================================


def get_action_result(state, action):
    if action == 'Stay':
        return state
    for n_state, n_action in get_neighbors(state):
        if n_action == action:
            return n_state
    return None

class CSP:
    def __init__(self, start_state, max_steps):
        self.start_state = start_state
        self.variables = list(range(max_steps))
        self.domains = {var: ['Up', 'Down', 'Left', 'Right', 'Stay'] for var in self.variables}
        self.max_steps = max_steps
        self.log_callback = None
        self.cancel_event = None

    def log(self, text):
        if self.log_callback:
            self.log_callback(text)

    def is_cancelled(self):
        return self.cancel_event and self.cancel_event.is_set()

    def get_state(self, assignment):
        state = self.start_state
        for i in range(len(assignment)):
            if i not in assignment:
                break
            action = assignment[i]
            next_state = get_action_result(state, action)
            if next_state is None:
                return None
            state = next_state
        return state

    def check_constraints(self, var, value, assignment):
        temp = assignment.copy()
        temp[var] = value

        # Binary constraint: no reverse
        if var > 0 and (var - 1) in assignment:
            prev_val = assignment[var - 1]
            opposites = {'Up': 'Down', 'Down': 'Up', 'Left': 'Right', 'Right': 'Left'}
            if value == opposites.get(prev_val):
                return False
            if prev_val == 'Stay' and value != 'Stay':
                return False

        # Validity constraint (Sequential)
        is_contiguous = all(i in temp for i in range(var + 1))
        if is_contiguous:
            state = self.start_state
            for i in range(var + 1):
                act = temp[i]
                if act == 'Stay' and state != GOAL:
                    return False
                if act != 'Stay' and state == GOAL:
                    return False

                n_state = get_action_result(state, act)
                if n_state is None:
                    return False
                state = n_state

        # Global constraint
        if len(temp) == self.max_steps:
            if self.get_state(temp) != GOAL:
                return False

        return True


# 1. Backtracking Search
def select_unassigned_variable(assignment, csp):
    for var in csp.variables:
        if var not in assignment:
            return var
    return None

def backtracking_search(csp):
    return recursive_backtracking({}, csp)

def recursive_backtracking(assignment, csp):
    if csp.is_cancelled(): return "failure"
    if len(assignment) == csp.max_steps:
        return assignment

    var = select_unassigned_variable(assignment, csp)
    for value in csp.domains[var]:
        if csp.check_constraints(var, value, assignment):
            assignment[var] = value
            csp.log(f"StepLog: [Backtracking] Assign var {var} = {value}")
            result = recursive_backtracking(assignment, csp)
            if result != "failure":
                return result
            del assignment[var]
    return "failure"


# 2. Forward Checking Search
def forward_checking_search(csp):
    return fc_recursive({}, csp)

def fc_recursive(assignment, csp):
    if csp.is_cancelled(): return "failure"
    if len(assignment) == csp.max_steps:
        return assignment

    var = select_unassigned_variable(assignment, csp)
    for value in list(csp.domains[var]):
        if csp.check_constraints(var, value, assignment):
            assignment[var] = value
            csp.log(f"StepLog: [FC] Assign var {var} = {value}")

            removals = []
            failed = False
            for unassigned in csp.variables:
                if unassigned not in assignment:
                    for val in list(csp.domains[unassigned]):
                        if not csp.check_constraints(unassigned, val, assignment):
                            csp.domains[unassigned].remove(val)
                            removals.append((unassigned, val))
                            csp.log(f"StepLog: [FC] Pruned var {unassigned} = {val}")

                    if len(csp.domains[unassigned]) == 0:
                        failed = True
                        break

            if not failed:
                result = fc_recursive(assignment, csp)
                if result != "failure":
                    return result

            del assignment[var]
            for u, val in removals:
                csp.domains[u].append(val)

    return "failure"


# 3. AC-3
def check_binary(xi, x, xj, y):
    opposites = {'Up': 'Down', 'Down': 'Up', 'Left': 'Right', 'Right': 'Left'}
    if xi == xj - 1 or xj == xi - 1:
        if x == opposites.get(y):
            return False

        if xi == xj - 1:
            if x == 'Stay' and y != 'Stay': return False
        if xj == xi - 1:
            if y == 'Stay' and x != 'Stay': return False

    return True

def rm_inconsistent_values(xi, xj, csp):
    removed = False
    for x in list(csp.domains[xi]):
        has_support = False
        for y in csp.domains[xj]:
            if check_binary(xi, x, xj, y):
                has_support = True
                break
        if not has_support:
            csp.domains[xi].remove(x)
            removed = True
            csp.log(f"StepLog: [AC-3] Pruned var {xi} = {x}")
    return removed

def ac3(csp):
    queue = []
    for i in range(csp.max_steps - 1):
        queue.append((i, i+1))
        queue.append((i+1, i))

    while queue:
        if csp.is_cancelled(): return False
        xi, xj = queue.pop(0)
        if rm_inconsistent_values(xi, xj, csp):
            if len(csp.domains[xi]) == 0:
                return False
            for xk in csp.variables:
                if xk != xi and xk != xj:
                    queue.append((xk, xi))
    return True

def ac3_search(csp):
    csp.log("StepLog: Running AC-3 to filter domains...")
    success = ac3(csp)
    if not success:
        csp.log("StepLog: AC-3 found no consistent domains.")
        return "failure"
    csp.log("StepLog: AC-3 finished. Proceeding with Backtracking...")
    return backtracking_search(csp)


# 4. Min-Conflicts
def count_conflicts(var, val, assignment, csp):
    temp = assignment.copy()
    temp[var] = val

    state = csp.start_state
    illegal_moves = 0
    reverses = 0

    for i in range(csp.max_steps):
        action = temp[i]
        if i > 0 and action == {'Up':'Down','Down':'Up','Left':'Right','Right':'Left'}.get(temp[i-1]):
            reverses += 1

        if state is not None:
            if action == 'Stay' and state != GOAL:
                illegal_moves += 1
            elif action != 'Stay' and state == GOAL:
                illegal_moves += 1

            n_state = get_action_result(state, action)
            if n_state is None:
                illegal_moves += 1
                state = None
            else:
                state = n_state
        else:
            illegal_moves += 1

    dist = 0
    if state is not None:
        for i in range(9):
            if state[i] != 0:
                gx, gy = divmod(GOAL.index(state[i]), 3)
                sx, sy = divmod(i, 3)
                dist += abs(gx - sx) + abs(gy - sy)
    else:
        dist = 100

    return illegal_moves * 50 + reverses * 10 + dist

def get_conflicted_vars(assignment, csp):
    conflicts = set()
    state = csp.start_state

    for i in range(csp.max_steps):
        action = assignment[i]
        if i > 0 and action == {'Up':'Down','Down':'Up','Left':'Right','Right':'Left'}.get(assignment[i-1]):
            conflicts.add(i)
            conflicts.add(i-1)

        if state is not None:
            if action == 'Stay' and state != GOAL:
                conflicts.add(i)
            elif action != 'Stay' and state == GOAL:
                conflicts.add(i)

            n_state = get_action_result(state, action)
            if n_state is None:
                conflicts.add(i)
                state = None
            else:
                state = n_state
        else:
            conflicts.add(i)

    if state != GOAL:
        if not conflicts:
            return list(range(csp.max_steps))

    return list(conflicts)

def min_conflicts(csp, max_steps=2000):
    current = {}
    for var in csp.variables:
        current[var] = random.choice(csp.domains[var])

    for i in range(max_steps):
        if csp.is_cancelled(): return "failure"

        conflicted_vars = get_conflicted_vars(current, csp)
        if not conflicted_vars and csp.get_state(current) == GOAL:
            return current

        if not conflicted_vars:
            conflicted_vars = list(range(csp.max_steps))

        var = random.choice(conflicted_vars)

        min_c = float('inf')
        best_vals = []
        for val in csp.domains[var]:
            c = count_conflicts(var, val, current, csp)
            if c < min_c:
                min_c = c
                best_vals = [val]
            elif c == min_c:
                best_vals.append(val)

        val = random.choice(best_vals)
        current[var] = val
        if i % 100 == 0:
            csp.log(f"StepLog: [Min-Conflicts] Iteration {i}, Var {var} -> {val}, Conflicts/Cost -> {min_c}")

    return "failure"


class CSPSearchBase(PuzzleSearchBase):
    def _process_result(self, assignment, max_steps, log_callback):
        if assignment == "failure" or not assignment:
            if log_callback: log_callback("StepLog: CSP failed to find a valid assignment of actions.")
            return None, [], set()

        curr = Node(self.start_state)
        curr.depth = 0
        curr.cost = 0

        for i in range(max_steps):
            action = assignment[i]
            if action == 'Stay':
                continue
            n_state = get_action_result(curr.state, action)
            child = Node(n_state, curr, action, curr.cost + 1)
            child.depth = curr.depth + 1
            curr = child

        if log_callback: log_callback("StepLog: CSP successfully found a valid assignment!")
        return curr, [], set()

# ========================================
# FILE: algorithms/csp/backtracking.py
# ========================================


class Backtracking(CSPSearchBase):
    def solve(self, log_callback=None, cancel_event=None):
        max_steps = self.param_val if self.param_val else 3
        csp = CSP(self.start_state, max_steps)
        csp.log_callback = log_callback
        csp.cancel_event = cancel_event

        assignment = backtracking_search(csp)
        return self._process_result(assignment, max_steps, log_callback)

# ========================================
# FILE: algorithms/csp/forward_checking.py
# ========================================


class ForwardChecking(CSPSearchBase):
    def solve(self, log_callback=None, cancel_event=None):
        max_steps = self.param_val if self.param_val else 3
        csp = CSP(self.start_state, max_steps)
        csp.log_callback = log_callback
        csp.cancel_event = cancel_event

        assignment = forward_checking_search(csp)
        return self._process_result(assignment, max_steps, log_callback)

# ========================================
# FILE: algorithms/csp/ac3.py
# ========================================


class AC3(CSPSearchBase):
    def solve(self, log_callback=None, cancel_event=None):
        max_steps = self.param_val if self.param_val else 3
        csp = CSP(self.start_state, max_steps)
        csp.log_callback = log_callback
        csp.cancel_event = cancel_event

        assignment = ac3_search(csp)
        return self._process_result(assignment, max_steps, log_callback)

# ========================================
# FILE: algorithms/csp/min_conflicts.py
# ========================================


class MinConflicts(CSPSearchBase):
    def solve(self, log_callback=None, cancel_event=None):
        max_steps = self.param_val if self.param_val else 3
        csp = CSP(self.start_state, max_steps)
        csp.log_callback = log_callback
        csp.cancel_event = cancel_event

        assignment = min_conflicts(csp, max_steps=1000)
        return self._process_result(assignment, max_steps, log_callback)

# ========================================
# FILE: algorithms/adversarial/adversarial_search.py
# ========================================


def heuristic(state):
    if state == GOAL:
        return 1000
    dist = 0
    for i in range(9):
        if state[i] != 0:
            gx, gy = divmod(GOAL.index(state[i]), 3)
            sx, sy = divmod(i, 3)
            dist += abs(gx - sx) + abs(gy - sy)
    return -dist

OPPOSITES = {'Up': 'Down', 'Down': 'Up', 'Left': 'Right', 'Right': 'Left'}

# --- MINIMAX ---
def minimax_decision(state, max_depth, cancel_event):
    v, action, pv = max_value(state, 0, max_depth, None, cancel_event)
    return pv

def max_value(state, depth, max_depth, prev_act, cancel_event):
    if cancel_event and cancel_event.is_set(): return -float('inf'), None, []
    if depth == max_depth or state == GOAL:
        return heuristic(state), None, []

    v = -float('inf')
    best_act = None
    best_pv = []

    for n_state, act in get_neighbors(state):
        if act == OPPOSITES.get(prev_act): continue
        v2, _, pv2 = min_value(n_state, depth + 1, max_depth, act, cancel_event)
        if v2 > v:
            v = v2
            best_act = act
            best_pv = [(act, n_state, 'MAX')] + pv2

    return v, best_act, best_pv

def min_value(state, depth, max_depth, prev_act, cancel_event):
    if cancel_event and cancel_event.is_set(): return float('inf'), None, []
    if depth == max_depth or state == GOAL:
        return heuristic(state), None, []

    v = float('inf')
    best_act = None
    best_pv = []

    for n_state, act in get_neighbors(state):
        if act == OPPOSITES.get(prev_act): continue
        v2, _, pv2 = max_value(n_state, depth + 1, max_depth, act, cancel_event)
        if v2 < v:
            v = v2
            best_act = act
            best_pv = [(act, n_state, 'MIN')] + pv2

    return v, best_act, best_pv


# --- ALPHA-BETA ---
def alpha_beta_search(state, max_depth, cancel_event):
    v, action, pv = ab_max_value(state, 0, max_depth, -float('inf'), float('inf'), None, cancel_event)
    return pv

def ab_max_value(state, depth, max_depth, alpha, beta, prev_act, cancel_event):
    if cancel_event and cancel_event.is_set(): return -float('inf'), None, []
    if depth == max_depth or state == GOAL:
        return heuristic(state), None, []

    v = -float('inf')
    best_act = None
    best_pv = []

    for n_state, act in get_neighbors(state):
        if act == OPPOSITES.get(prev_act): continue
        v2, _, pv2 = ab_min_value(n_state, depth + 1, max_depth, alpha, beta, act, cancel_event)
        if v2 > v:
            v = v2
            best_act = act
            best_pv = [(act, n_state, 'MAX')] + pv2
        if v >= beta:
            return v, best_act, best_pv
        alpha = max(alpha, v)

    return v, best_act, best_pv

def ab_min_value(state, depth, max_depth, alpha, beta, prev_act, cancel_event):
    if cancel_event and cancel_event.is_set(): return float('inf'), None, []
    if depth == max_depth or state == GOAL:
        return heuristic(state), None, []

    v = float('inf')
    best_act = None
    best_pv = []

    for n_state, act in get_neighbors(state):
        if act == OPPOSITES.get(prev_act): continue
        v2, _, pv2 = ab_max_value(n_state, depth + 1, max_depth, alpha, beta, act, cancel_event)
        if v2 < v:
            v = v2
            best_act = act
            best_pv = [(act, n_state, 'MIN')] + pv2
        if v <= alpha:
            return v, best_act, best_pv
        beta = min(beta, v)

    return v, best_act, best_pv


# --- EXPECTIMAX ---
def expectimax_search(state, max_depth, cancel_event):
    v, action, pv = exp_max_value(state, 0, max_depth, None, cancel_event)
    return pv

def exp_max_value(state, depth, max_depth, prev_act, cancel_event):
    if cancel_event and cancel_event.is_set(): return -float('inf'), None, []
    if depth == max_depth or state == GOAL:
        return heuristic(state), None, []

    v = -float('inf')
    best_act = None
    best_pv = []

    for n_state, act in get_neighbors(state):
        if act == OPPOSITES.get(prev_act): continue
        v2, pv2 = exp_value(n_state, depth + 1, max_depth, act, cancel_event)
        if v2 > v:
            v = v2
            best_act = act
            best_pv = [(act, n_state, 'MAX')] + pv2

    return v, best_act, best_pv

def exp_value(state, depth, max_depth, prev_act, cancel_event):
    if cancel_event and cancel_event.is_set(): return 0, []
    if depth == max_depth or state == GOAL:
        return heuristic(state), []

    neighbors = [(s, a) for s, a in get_neighbors(state) if a != OPPOSITES.get(prev_act)]
    if not neighbors:
        return heuristic(state), []

    v = 0
    prob = 1.0 / len(neighbors)
    best_pv = []
    max_v2 = -float('inf')

    for n_state, act in neighbors:
        v2, _, pv2 = exp_max_value(n_state, depth + 1, max_depth, act, cancel_event)
        v += prob * v2
        if v2 > max_v2:
            max_v2 = v2
            best_pv = [(act, n_state, 'CHANCE')] + pv2

    return v, best_pv


# --- BASE CLASS ---
class AdversarialSearchBase(PuzzleSearchBase):
    def _process_pv(self, pv, log_callback):
        if not pv:
            if log_callback: log_callback("StepLog: No path found or depth limit is 0.")
            return None, [], set()

        curr = Node(self.start_state)
        curr.depth = 0
        curr.cost = 0

        for act, n_state, player in pv:
            if log_callback: log_callback(f"StepLog: [{player}] predicted action: {act}")
            child = Node(n_state, curr, act, curr.cost + 1)
            child.depth = curr.depth + 1
            child.action = f"[{player}] {act}"
            curr = child

        if log_callback: log_callback("StepLog: Principal Variation (PV) extracted successfully.")
        return curr, [], set()

# ========================================
# FILE: algorithms/adversarial/minimax.py
# ========================================


class Minimax(AdversarialSearchBase):
    def solve(self, log_callback=None, cancel_event=None):
        max_depth = self.param_val if self.param_val else 5
        if log_callback: log_callback(f"StepLog: Running Minimax (MAX vs MIN) with max_depth={max_depth}...")
        pv = minimax_decision(self.start_state, max_depth, cancel_event)
        return self._process_pv(pv, log_callback)

# ========================================
# FILE: algorithms/adversarial/alpha_beta.py
# ========================================


class AlphaBeta(AdversarialSearchBase):
    def solve(self, log_callback=None, cancel_event=None):
        max_depth = self.param_val if self.param_val else 5
        if log_callback: log_callback(f"StepLog: Running Alpha-Beta Pruning with max_depth={max_depth}...")
        pv = alpha_beta_search(self.start_state, max_depth, cancel_event)
        return self._process_pv(pv, log_callback)

# ========================================
# FILE: algorithms/adversarial/expectimax.py
# ========================================


class Expectimax(AdversarialSearchBase):
    def solve(self, log_callback=None, cancel_event=None):
        max_depth = self.param_val if self.param_val else 5
        if log_callback: log_callback(f"StepLog: Running Expectimax (MAX vs CHANCE) with max_depth={max_depth}...")
        pv = expectimax_search(self.start_state, max_depth, cancel_event)
        return self._process_pv(pv, log_callback)

# ========================================
# FILE: main.py
# ========================================




ALGO_GROUPS = [
    ("Uninformed Search",       ["BFS", "DFS", "UCS", "IDS"]),
    ("Informed Search",         ["Greedy Best-First", "A*"]),
    ("Local Search",            ["Simple Hill Climbing", "Steepest-Ascent Hill Climbing", "Stochastic Hill Climbing", "Simulated Annealing", "Local Beam Search (k=3)"]),
    ("Complex Environment",    ["Belief-State BFS", "Partial Observation", "AND-OR Graph Search"]),
    ("Constraint Satisfaction", ["CSP: Backtracking", "CSP: Forward Checking", "CSP: AC-3", "CSP: Min-Conflicts"]),
    ("Adversarial Search",     ["Minimax", "Alpha-Beta Pruning", "Expectimax"]),
]


ALGORITHMS = {
    "BFS": BFS, "DFS": DFS, "UCS": UCS, "IDS": IDS, "Greedy Best-First": GreedyBestFirstSearch, "A*": AStar,
    "Simple Hill Climbing": SimpleHillClimbing, "Steepest-Ascent Hill Climbing": SteepestAscentHillClimbing,
    "Stochastic Hill Climbing": StochasticHillClimbing, "Simulated Annealing": SimulatedAnnealing,
    "Local Beam Search (k=3)": LocalBeamSearch, "Belief-State BFS": BeliefStateBFS,
    "Partial Observation": PartialObservationSearch, "AND-OR Graph Search": AndOrGraphSearch,
    "CSP: Backtracking": Backtracking, "CSP: Forward Checking": ForwardChecking, "CSP: AC-3": AC3,
    "CSP: Min-Conflicts": MinConflicts, "Backtracking": Backtracking, "Forward Checking": ForwardChecking,
    "Minimax (Tic-Tac-Toe)": Minimax, "Alpha-Beta (Tic-Tac-Toe)": AlphaBeta, "Minimax": Minimax,
    "Alpha-Beta Pruning": AlphaBeta, "Expectimax": Expectimax
}

class EightPuzzleApp(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("8-Puzzle Search Algorithms")
        self.configure(bg="#F9FAFB")
        self.resizable(True, True)
        self.minsize(1100, 780)

        self.current_state = list(GOAL)
        self.path = []
        self.current_step_idx = 0
        self.is_running = False
        self.is_animating = False

        self._build_ui()
        self._reset()

    def _build_ui(self):
        # Top Control Bar
        top_controls = tk.Frame(self, bg="#FFFFFF", pady=8, padx=14)
        top_controls.pack(fill="x")

        tk.Label(top_controls, text="8-PUZZLE SEARCH ALGORITHMS", bg="#FFFFFF", fg="#1E293B", font=("Segoe UI", 16, "bold")).pack(side="left", padx=(0, 20))

        tk.Label(top_controls, text="Group:", bg="#FFFFFF", fg="#475569", font=("Segoe UI", 9)).pack(side="left", padx=(10, 2))
        self._group_var = tk.StringVar(value=ALGO_GROUPS[0][0])
        group_cb = ttk.Combobox(top_controls, textvariable=self._group_var, values=[g[0] for g in ALGO_GROUPS], state="readonly", width=18)
        group_cb.pack(side="left", padx=4)

        tk.Label(top_controls, text="Algorithm:", bg="#FFFFFF", fg="#475569", font=("Segoe UI", 9)).pack(side="left", padx=(15, 2))
        self._algo_var = tk.StringVar(value=ALGO_GROUPS[0][1][0])
        self.algo_cb = ttk.Combobox(top_controls, textvariable=self._algo_var, values=ALGO_GROUPS[0][1], state="readonly", width=22)
        self.algo_cb.pack(side="left", padx=4)

        def on_group_change(e):
            group_name = self._group_var.get()
            for g, algos in ALGO_GROUPS:
                if g == group_name:
                    self.algo_cb.config(values=algos)
                    self.algo_cb.current(0)
                    self._on_algo_change()
                    break
        group_cb.bind("<<ComboboxSelected>>", on_group_change)
        self.algo_cb.bind("<<ComboboxSelected>>", self._on_algo_change)

        tk.Label(top_controls, text="Parameter:", bg="#FFFFFF", fg="#475569", font=("Segoe UI", 9)).pack(side="left", padx=(15, 2))
        self.param_entry = tk.Spinbox(top_controls, from_=1, to=200, width=5)
        self.param_entry.delete(0, "end")
        self.param_entry.insert(0, "35")
        self.param_entry.pack(side="left", padx=2)

        btn_cfg = dict(font=("Segoe UI", 9), relief="solid", cursor="hand2", pady=2, padx=12, bd=1)
        self._make_btn(top_controls, "Random", "#FFFFFF", "#1E293B", self._shuffle, side="left", pack_padx=(20, 4), **btn_cfg)

        self._btn_run = self._make_btn(top_controls, "Run", "#2563EB", "#FFFFFF", self._run, side="left", pack_padx=4, **btn_cfg)
        self._btn_run.config(bd=0, font=("Segoe UI", 9, "bold")) # Make Run button stand out

        self._btn_stop_algo = self._make_btn(top_controls, "Stop", "#EF4444", "#FFFFFF", self._stop_algo, state="disabled", side="left", pack_padx=4, **btn_cfg)
        self._btn_stop_algo.config(bd=0, font=("Segoe UI", 9, "bold"))

        self._make_btn(top_controls, "Reset", "#FFFFFF", "#1E293B", self._reset, side="right", pack_padx=10, **btn_cfg)

        # Main Layout
        middle_frame = tk.Frame(self, bg="#F9FAFB")
        middle_frame.pack(fill="x", padx=10, pady=10)

        # Left Panel (Goal + Board + Controls)
        left_panel = tk.Frame(middle_frame, bg="#F9FAFB")
        left_panel.pack(side="left", fill="both", expand=True)

        board_container = tk.Frame(left_panel, bg="#F9FAFB")
        board_container.pack(fill="x", pady=10)

        goal_frame = tk.LabelFrame(board_container, text="Goal State", bg="#F9FAFB", fg="#475569")
        goal_frame.pack(side="left", anchor="n", padx=(10, 20))
        goal_inner = tk.Frame(goal_frame, bg="#F9FAFB")
        goal_inner.pack(padx=5, pady=5)
        for i in range(9):
            val = GOAL[i]
            tk.Label(goal_inner, text=str(val) if val!=0 else "-", font=("Segoe UI", 10), width=2, bg="#F9FAFB").grid(row=i//3, column=i%3)

        self.canvas = PuzzleCanvas(board_container)
        self.canvas.pack(side="left")

        nav_frame = tk.Frame(left_panel, bg="#F9FAFB")
        nav_frame.pack(fill="x", pady=10, padx=10)
        tk.Button(nav_frame, text="◀ Prev", command=self._prev_step, **btn_cfg).pack(side="left", padx=4)
        tk.Button(nav_frame, text="Next ▶", command=self._next_step, **btn_cfg).pack(side="left", padx=4)
        self._btn_stop = tk.Button(nav_frame, text="Auto Play", command=self._start_animation, **btn_cfg)
        self._btn_stop.pack(side="left", padx=(4, 20))

        tk.Label(nav_frame, text="Speed", bg="#F9FAFB", font=("Segoe UI", 9)).pack(side="left", padx=2)
        self.speed_scale = tk.Scale(nav_frame, from_=50, to=1000, orient="horizontal", bg="#F9FAFB", bd=0, showvalue=1, resolution=50, length=200, highlightthickness=0)
        self.speed_scale.set(300)
        self.speed_scale.pack(side="left")

        # Right Panel (Stats)
        right_panel = tk.Frame(middle_frame, bg="#F9FAFB", width=280)
        right_panel.pack(side="right", fill="y", padx=(10, 0))
        right_panel.pack_propagate(False)
        self.stats = StatsPanel(right_panel)
        self.stats.pack(fill="both", expand=True)

        # Bottom Frame (Notebook)
        bottom_frame = tk.Frame(self, bg="#F9FAFB")
        bottom_frame.pack(fill="both", expand=True, padx=10, pady=(0, 10))
        self.notebook = DetailsNotebook(bottom_frame, on_solution_click_cb=self._jump_to_step)
        self.notebook.pack(fill="both", expand=True)

    def _make_btn(self, parent, text, bg, fg, cmd, side="left", pack_padx=0, pack_pady=0, **kw):
        btn = tk.Button(parent, text=text, bg=bg, fg=fg, command=cmd, **kw)
        btn.pack(side=side, padx=pack_padx, pady=pack_pady)
        return btn

    @staticmethod
    def _lighten(hex_color):
        try:
            r = min(255, int(hex_color[1:3], 16) + 40)
            g = min(255, int(hex_color[3:5], 16) + 40)
            b = min(255, int(hex_color[5:7], 16) + 40)
            return f"#{r:02x}{g:02x}{b:02x}"
        except:
            return hex_color

    def _on_algo_change(self, e=None):
        algo_name = self._algo_var.get()
        group_name = self._group_var.get()

        self.param_entry.delete(0, "end")
        if group_name == "Constraint Satisfaction":
            self.param_entry.insert(0, "4")
        elif group_name == "Adversarial Search":
            self.param_entry.insert(0, "11")
        elif group_name == "Local Search":
            if algo_name == "Simulated Annealing":
                self.param_entry.insert(0, "100")
            elif algo_name == "Local Beam Search (k=3)":
                self.param_entry.insert(0, "3")
            else:
                self.param_entry.insert(0, "50")
        else:
            self.param_entry.insert(0, "35")

        if hasattr(self.notebook, 'rename_tab_text'):
            if group_name == "Informed Search":
                self.notebook.rename_tab_text("Frontier", "Reached")
            else:
                self.notebook.rename_tab_text("Frontier", "Frontier")

        if algo_name == "Belief-State BFS":
            state1 = tuple(self.current_state)
            state2 = list(state1)
            for _ in range(5):
                state2 = list(random.choice(get_neighbors(tuple(state2)))[0])
            self.canvas.update_grid((state1, tuple(state2)))
        elif algo_name == "Partial Observation":
            st = list(self.current_state)
            hidden = 0
            for i in range(8, -1, -1):
                if st[i] != 0:
                    st[i] = -1
                    hidden += 1
                    if hidden == 2: break
            self.canvas.update_grid((tuple(st),))
        elif algo_name == "AND-OR Graph Search":
            state = list(GOAL)
            for _ in range(3):
                state = list(random.choice(get_neighbors(tuple(state)))[0])
            self.current_state = state
            self.canvas.update_grid(tuple(state))
        elif algo_name.startswith("CSP:"):
            from collections import deque
            queue = deque([(tuple(GOAL), 0)])
            visited = {tuple(GOAL)}
            states_at_depth = []
            while queue:
                st, d = queue.popleft()
                if d == 4:
                    states_at_depth.append(list(st))
                    continue
                if d > 4: break
                for n_st, _ in get_neighbors(st):
                    if n_st not in visited:
                        visited.add(n_st)
                        queue.append((n_st, d + 1))
            self.current_state = random.choice(states_at_depth) if states_at_depth else list(GOAL)
            self.canvas.update_grid(tuple(self.current_state))
        elif group_name == "Adversarial Search":
            from collections import deque
            queue = deque([(tuple(GOAL), 0)])
            visited = {tuple(GOAL)}
            states_at_depth = []
            while queue:
                st, d = queue.popleft()
                if d == 10:
                    states_at_depth.append(list(st))
                    continue
                if d > 10: break
                for n_st, _ in get_neighbors(st):
                    if n_st not in visited:
                        visited.add(n_st)
                        queue.append((n_st, d + 1))
            self.current_state = random.choice(states_at_depth) if states_at_depth else list(GOAL)
            self.canvas.update_grid(tuple(self.current_state))
            self.after(100, lambda: self.notebook.write_tab("Step Log", "LƯU Ý: 8-Puzzle là bài toán 1 người chơi (Single-agent). Việc áp dụng Minimax/Alpha-Beta ở đây là tạo ra một luật chơi giả định (2 người chơi luân phiên: MAX cố giải, MIN cố phá) chỉ nhằm mục đích minh họa cách duyệt cây đối kháng!"))
        else:
            self.canvas.update_grid(self.current_state)

    def _reset(self):
        self._stop_animation()
        self.current_state = list(GOAL)
        self.path = []
        self.current_step_idx = 0
        self.canvas.update_grid(self.current_state)
        self.notebook.clear_all()
        self.stats.reset_current()
        self.stats.update_stat("algo", self._algo_var.get())

    def _shuffle(self):
        self._reset()
        state = list(GOAL)
        for _ in range(14):
            state = list(random.choice(get_neighbors(tuple(state)))[0])
        self.current_state = state
        self._on_algo_change()

    def _stop_animation(self):
        self.is_animating = False
        if hasattr(self, '_btn_stop'):
            self._btn_stop.config(state="disabled")

    def _stop_algo(self):
        if self.is_running and hasattr(self, '_cancel_event'):
            self._cancel_event.set()
            self._btn_stop_algo.config(state="disabled")
            self._safe_log("StepLog: Đã gửi lệnh hủy tìm kiếm...")

    def _safe_log(self, text, tab="Step Log"):
        self.after(0, lambda: self.notebook.write_tab(tab, text))

    def _safe_log_routing(self, text):
        if text.startswith("Node: "):
            self._safe_log(text[6:], "Nodes")
        elif text.startswith("StepLog: "):
            self._safe_log(text[9:], "Step Log")
        elif text.startswith("Conditional Plan:"):
            # Strip "Conditional Plan:\n" if it's there, or just print it.
            # and_or_search.py sends: "Conditional Plan:\n..."
            content = text.replace("Conditional Plan:\n", "")
            if content == "Conditional Plan: FAILURE (Depth limit reached or no path)":
                self._safe_log("FAILURE (Depth limit reached or no path)", "Conditional Plan")
            else:
                self._safe_log(content, "Conditional Plan")
        else:
            self._safe_log(text, "Step Log")

    def _run(self):
        if self.is_running or self.is_animating: return
        self.is_running = True
        self._cancel_event = threading.Event()
        self._btn_run.config(state="disabled")
        self._btn_stop.config(state="disabled")
        self._btn_stop_algo.config(state="normal")
        self.notebook.clear_all()
        if hasattr(self.notebook, 'rename_tab_text'):
            self.notebook.rename_tab_text("Solution Path", "Solution Path")

        algo_name = self._algo_var.get()
        self.stats.update_stat("algo", algo_name)
        self.stats.update_status("Đang tìm kiếm...", "#F38BA8")

        try:
            param = int(self.param_entry.get())
        except:
            param = 35

        algo_cls = ALGORITHMS.get(algo_name)
        if not algo_cls:
            self.is_running = False
            self._btn_run.config(state="normal")
            return

        if algo_name == "Belief-State BFS":
            # Generate a belief state with 2 elements
            state1 = tuple(self.current_state)
            state2 = list(state1)
            for _ in range(5):
                state2 = list(random.choice(get_neighbors(tuple(state2)))[0])
            start_tuple = (state1, tuple(state2))
            self.canvas.update_grid(start_tuple)
        elif algo_name == "Partial Observation":
            # Hide 2 tiles
            st = list(self.current_state)
            hidden = 0
            for i in range(8, -1, -1):
                if st[i] != 0:
                    st[i] = -1
                    hidden += 1
                    if hidden == 2: break
            start_tuple = tuple(st)
            self.canvas.update_grid(start_tuple)
        else:
            start_tuple = tuple(self.current_state)
            self.canvas.update_grid(start_tuple)

        solver = algo_cls(start_tuple, param_val=param)

        threading.Thread(target=self._solve_thread, args=(solver,), daemon=True).start()

    def _solve_thread(self, solver):
        t0 = time.time()
        result_node, frontier, explored = solver.solve(log_callback=self._safe_log_routing, cancel_event=self._cancel_event)
        t1 = time.time()

        self.after(0, lambda: self._on_solve_done(result_node, frontier, explored, t1 - t0))

    def _on_solve_done(self, result_node, frontier, explored, time_taken):
        self.is_running = False
        self._btn_run.config(state="normal")
        self._btn_stop_algo.config(state="disabled")

        if self._cancel_event.is_set():
            self.stats.update_status(f"Đã hủy ({time_taken:.2f}s)", "#F38BA8")
        else:
            is_goal_reached = result_node and getattr(result_node, 'state', None) == tuple(GOAL)
            if is_goal_reached or self._algo_var.get() == "Partial Observation" or self._algo_var.get() == "Belief-State BFS":
                self.stats.update_status(f"Hoàn thành ({time_taken:.2f}s)")
            else:
                self.stats.update_status(f"Kẹt ở Local Max ({time_taken:.2f}s)", "#F59E0B")
                if hasattr(self.notebook, 'rename_tab_text'):
                    self.notebook.rename_tab_text("Solution Path", "Path to Local Max")
        if explored:
            self.stats.update_stat("explored", len(explored))
        if frontier:
            self.stats.update_stat("frontier", len(frontier))

        def format_state_row(states):
            if not states: return ""
            if isinstance(states[0], (list, tuple)) and isinstance(states[0][0], (list, tuple)):
                blocks = []
                for st in states:
                    formatted = [format_state_matrix(s).split('\n') for s in st]
                    rows = []
                    for i in range(3):
                        rows.append(" | ".join(f[i] for f in formatted))
                    blocks.append(rows)
                res_rows = []
                for i in range(3):
                    res_rows.append("    ".join(b[i] for b in blocks))
                return "\n".join(res_rows)
            else:
                blocks = [format_state_matrix(s).split('\n') for s in states]
                res_rows = []
                for i in range(3):
                    res_rows.append("    ".join(b[i] for b in blocks))
                return "\n".join(res_rows)

        def print_states(tab_name, state_list, limit=30):
            d_name = "Reached" if tab_name == "Frontier" and self._group_var.get() == "Informed Search" else tab_name
            self.notebook.write_tab(tab_name, f"{d_name} ({len(state_list)} states):")
            to_print = state_list[:limit]

            items_per_row = 6
            if to_print and isinstance(to_print[0], (list, tuple)) and isinstance(to_print[0][0], (list, tuple)):
                items_per_row = max(1, 10 // (len(to_print[0]) + 1))

            for i in range(0, len(to_print), items_per_row):
                chunk = to_print[i:i+items_per_row]
                self.notebook.write_tab(tab_name, format_state_row(chunk) + "\n")
            if len(state_list) > limit:
                self.notebook.write_tab(tab_name, f"... and {len(state_list)-limit} more")

        if frontier:
            print_states("Frontier", [f.state for f in frontier])

        if explored:
            print_states("Explored", list(explored))

        path = []
        curr = result_node
        while curr:
            path.append(curr)
            curr = getattr(curr, 'parent', None)
        path.reverse()

        self.path = path
        if path:
            self.stats.update_stat("path_cost", getattr(result_node, 'cost', 0))
            self.stats.update_stat("depth", getattr(result_node, 'depth', 0))
            self.notebook.render_solution_path(path)
            self._jump_to_step(0)
            self._start_animation()
        else:
            self.notebook.write_tab("Solution Path", "No path found or algorithm is a placeholder.")

    def _start_animation(self):
        if len(self.path) <= 1:
            return
        self.is_animating = True
        self._btn_stop.config(state="normal")
        self._animate_step()

    def _animate_step(self):
        if not self.is_animating: return

        next_step = self.current_step_idx + 1
        if next_step < len(self.path):
            self._jump_to_step(next_step)
            self.after(300, self._animate_step)
        else:
            self._stop_animation()

    def _jump_to_step(self, step_idx):
        if not self.path or step_idx < 0 or step_idx >= len(self.path): return
        self.current_step_idx = step_idx
        node = self.path[step_idx]

        highlight_idx = None
        next_action_str = ""
        if step_idx < len(self.path) - 1:
            next_node = self.path[step_idx + 1]
            next_action_str = getattr(next_node, 'action', '')
            if isinstance(next_node.state, (list, tuple)) and isinstance(next_node.state[0], int):
                highlight_idx = next_node.state.index(0)
        else:
            next_action_str = "GAME OVER"

        # Update board and pass highlight info
        if hasattr(self.canvas, 'update_grid') and 'highlight_idx' in self.canvas.update_grid.__code__.co_varnames:
            self.canvas.update_grid(node.state, highlight_idx=highlight_idx, turn_str=next_action_str)
        else:
            self.canvas.update_grid(node.state)

        action_str = getattr(node, 'action', 'Start') if getattr(node, 'action', None) else "Start"
        self.stats.update_stat("step", f"{step_idx} / {len(self.path)-1}")
        self.stats.update_stat("action", action_str)

        if hasattr(self.canvas, 'set_turn'):
            self.canvas.set_turn(next_action_str)

        self.notebook.highlight_step(step_idx)

    def _next_step(self):
        self._stop_animation()
        self._jump_to_step(self.current_step_idx + 1)

    def _prev_step(self):
        self._stop_animation()
        self._jump_to_step(self.current_step_idx - 1)


if __name__ == "__main__":
    app = EightPuzzleApp()
    app.mainloop()
